In [ ]:
# Cell 1: Setup
!pip install -q google-genai fastapi uvicorn python-dotenv

import json, os, sys
from google import genai
from google.genai import types


In [ ]:
# Cell 2: Chispa core (self-contained inline)




import json
import os
from google import genai
from google.genai import types

SYSTEM_PROMPT = """You are Chispa â€” a warm, direct AI companion for working adults who are scared of AI.
Your only job is to guide this person to their first real win with AI in under 20 minutes.

Rules you never break:
1. Never use technical jargon. If a technical word is unavoidable, explain it immediately in plain language.
2. Detect the user's language from their first message. Respond in that language for the entire session. Never switch.
3. Ask exactly ONE question at a time. Never list multiple questions.
4. Never lecture. Never explain before the win. Knowledge comes AFTER the experience.
5. Be warm but efficient. You are a smart friend, not a teacher, not a chatbot, not a course.
6. If the user expresses fear or doubt, acknowledge it in one sentence, then move forward.
7. Never mention that you are an AI model or describe your technical architecture.

Session structure you follow silently:
DISCOVER -> PICK -> WIN -> PILL -> MAP
You know which stage you are in. The user does not need to know."""

MODEL = 'gemma-4-26b-a4b-it'
TEMPERATURE = 0.7
MAX_TOKENS = 1024


def build_client(api_key: str) -> genai.Client:
    return genai.Client(api_key=api_key)


def build_history(turns: list[dict]) -> list[types.Content]:
    return [
        types.Content(
            role=turn['role'],
            parts=[types.Part(text=turn['text'])]
        )
        for turn in turns
    ]


def _call(client: genai.Client, contents, response_json: bool = False) -> str:
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=TEMPERATURE,
        max_output_tokens=MAX_TOKENS,
        **({'response_mime_type': 'application/json'} if response_json else {}),
    )
    for attempt in range(2):
        response = client.models.generate_content(
            model=MODEL,
            config=config,
            contents=contents,
        )
        text = response.text or ''
        if text.strip():
            return text
    return ''


_GENERIC_PHRASES = [
    'save time', 'be more productive', 'increase efficiency',
    'improve workflow', 'work smarter', 'do more with less',
]


def _is_generic(use_cases: list) -> bool:
    combined = ' '.join(
        (uc.get('label', '') + ' ' + uc.get('description', '')).lower()
        for uc in use_cases
    )
    return any(phrase in combined for phrase in _GENERIC_PHRASES)


def run_discovery(client: genai.Client, conversation_history: list) -> dict:
    job_description = conversation_history[-1].parts[0].text

    base_prompt = f'''Input: {job_description}

The user just described their job. Your task:
1. Identify their role in 3 words or less (e.g. \"office administrator\", \"sales assistant\")
2. Generate exactly 3 concrete, specific AI use cases for that exact role. Not generic. Not abstract. Real tasks they do every week that AI can help with RIGHT NOW.
3. Frame each use case as a benefit the user gets, not a feature of AI.

Return ONLY valid JSON. No explanation. No preamble.

{{
  \"role\": \"string â€” their job role in 3 words max\",
  \"language\": \"string â€” ISO 639-1 code of the language they wrote in\",
  \"use_cases\": [
    {{\"id\": 1, \"label\": \"string â€” 4 words max, action-oriented\", \"description\": \"string â€” one sentence, plain language\"}},
    {{\"id\": 2, \"label\": \"string\", \"description\": \"string\"}},
    {{\"id\": 3, \"label\": \"string\", \"description\": \"string\"}}
  ]
}}'''

    for attempt in range(2):
        extra = ''
        if attempt == 1:
            extra = '\\nReturn ONLY valid JSON, no markdown, no backticks. Each use case must name a specific task they do, not a general benefit.'

        contents = list(conversation_history) + [
            types.Content(role='user', parts=[types.Part(text=base_prompt + extra)])
        ]
        raw = _call(client, contents, response_json=True)

        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, ValueError):
            if attempt == 0:
                continue
            raise ValueError(f'run_discovery: Gemma 4 returned invalid JSON after 2 attempts: {raw}')

        if _is_generic(data.get('use_cases', [])) and attempt == 0:
            continue

        return data

    raise ValueError('run_discovery: failed to get valid non-generic response')


_PILL_KEYWORDS = {
    1: ['write', 'draft', 'compose', 'email', 'letter', 'message', 'report'],
    2: ['summarize', 'summary', 'organize', 'structure', 'notes', 'recap'],
    3: ['share', 'upload', 'data', 'spreadsheet', 'document', 'analyze'],
    4: ['decide', 'approve', 'review', 'act', 'action'],
}


def select_pill(selected_use_case: dict) -> int:
    label = selected_use_case.get('label', '')
    description = selected_use_case.get('description', '')
    text = f"{label} {description}".lower()
    for pill_id in [2, 3, 4, 1]:
        if any(kw in text for kw in _PILL_KEYWORDS[pill_id]):
            return pill_id
    return 1


def run_pick_confirm(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    role: str,
    language: str,
) -> str:
    prompt = f'''Input: {selected_use_case['label']}, {role}, {language}

The user just picked their use case. Write one warm, encouraging sentence that:
- Confirms their choice
- Tells them they're about to do this right now, not learn about it
- Sounds like a smart friend, not a tutor

Respond in {language}. One sentence only. No questions.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def run_win_open(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    role: str,
    language: str,
) -> str:
    prompt = f'''Input: {selected_use_case}, {role}, {language}

The user is a {role}. They chose to work on: {selected_use_case['label']} â€” {selected_use_case['description']}.

Your job now: guide them to complete this task using AI right now.

Step 1: Ask them for the specific details you need to do this task FOR them.
- Ask for ONLY what is strictly necessary. One question maximum.
- Be specific. Not \"tell me more\" â€” ask for the exact input you need.

Respond in {language}. One question only.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def _quality_check(client: genai.Client, output: str, user_task_details: str, language: str) -> bool:
    prompt = f'''Score this AI output on 3 criteria. Return JSON {{\"pass\": true}} or {{\"pass\": false}}.

Criteria:
1. Is the output specific to these user details: \"{user_task_details}\"? (not generic filler)
2. Is it in language \"{language}\" with appropriate tone?
3. Would a real person use this as-is without major editing?

Output to score:
{output}'''

    config = types.GenerateContentConfig(
        temperature=0.1,
        max_output_tokens=50,
        response_mime_type='application/json',
    )
    response = client.models.generate_content(
        model=MODEL,
        config=config,
        contents=[types.Content(role='user', parts=[types.Part(text=prompt)])]
    )
    try:
        return json.loads(response.text or '{}').get('pass', True)
    except (json.JSONDecodeError, ValueError):
        return True


def run_win_execute(
    client: genai.Client,
    conversation_history: list,
    selected_use_case: dict,
    user_task_details: str,
    role: str,
    language: str,
) -> dict:
    base_prompt = f'''Input: {selected_use_case}, {user_task_details}, {role}, {language}

The user provided the details needed. Now do the task.
Complete the task fully and well. Do not explain what you are doing. Just do it.
After the output, add ONE short line asking if this looks good.

Respond in {language}.'''

    output = ''
    for attempt in range(2):
        extra = ''
        if attempt == 1:
            extra = '\\nThe previous output was too generic. Use the exact details provided. Make it specific, professional, and immediately usable.'

        contents = list(conversation_history) + [
            types.Content(role='user', parts=[types.Part(text=base_prompt + extra)])
        ]
        output = _call(client, contents)

        if attempt == 0 and not _quality_check(client, output, user_task_details, language):
            continue

        sentences = [s.strip() for s in output.split('.') if s.strip()]
        summary = '. '.join(sentences[:2]) + ('.' if sentences else '')
        return {'output': output, 'summary': summary}

    sentences = [s.strip() for s in output.split('.') if s.strip()]
    summary = '. '.join(sentences[:2]) + ('.' if sentences else '')
    return {'output': output, 'summary': summary}


def run_win_confirm(client: genai.Client, conversation_history: list, language: str) -> str:
    prompt = f'''Input: {language}

The user just confirmed their AI output looks good. This is their first win.
Write one sentence that celebrates this moment â€” warm, genuine, not over the top.
Then transition: tell them you want to share something quick about what just happened.

Respond in {language}. Two sentences maximum.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


_PILL_NAMES = {
    1: 'Prompting',
    2: 'AI strengths',
    3: 'Context',
    4: 'Hallucination',
}

_PILL_DEFINITIONS = {
    1: 'What a prompt is + when to be specific vs vague',
    2: 'What AI is genuinely good at + when NOT to use it',
    3: 'What context means in AI + how much to share at work',
    4: 'What hallucination is + when to verify AI output',
}


def run_pill(
    client: genai.Client,
    conversation_history: list,
    pill_id: int,
    selected_use_case: dict,
    role: str,
    language: str,
    task_output_summary: str,
) -> str:
    prompt = f'''Input: {pill_id}, {selected_use_case}, {role}, {language}, {task_output_summary}

Deliver Pill {pill_id} to this user. They are a {role} who just completed: {selected_use_case['label']}.

Pill definition: {_PILL_DEFINITIONS[pill_id]}

Format your pill EXACTLY like this:
1. One sentence naming the concept in plain language (no jargon)
2. One analogy drawn from their specific job/industry (not generic)
3. One question that connects this concept to something they already do at work

Do NOT use bullet points. Write it as natural speech.
Respond in {language}.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)


def run_map(
    client: genai.Client,
    conversation_history: list,
    role: str,
    selected_use_case: dict,
    pill_id: int,
    language: str,
) -> str:
    pill_concept = _PILL_NAMES.get(pill_id, 'Prompting')
    prompt = f'''Input: {role}, {selected_use_case}, {pill_concept}, {language}

The user is a {role}. They just completed their first AI task: {selected_use_case['label']}.
They learned about: {pill_concept}.

Generate their personal AI map: exactly 3 next steps they can take THIS WEEK.

Rules:
- Each step must be specific to their role. Not generic advice.
- Each step must be something they can do in under 30 minutes.
- Each step must build on what they just did â€” not start over.
- No jargon. No tool names they don't know yet. One free tool recommendation maximum per step.
- Format as numbered list. One sentence per step. Action verb to start.

Respond in {language}.'''

    contents = list(conversation_history) + [
        types.Content(role='user', parts=[types.Part(text=prompt)])
    ]
    return _call(client, contents)

In [ ]:
# Cell 3: API key
# On Kaggle: add GOOGLE_API_KEY as a Kaggle Secret
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['GOOGLE_API_KEY'] = secrets.get_secret('GOOGLE_API_KEY')

client = build_client(os.environ['GOOGLE_API_KEY'])
print('Client ready.')

In [ ]:
# Cell 4: Discovery
print('Hi! I\'m Chispa. I\'m here to help you do something real with AI â€” today, in the next 20 minutes.\n')
user_input = input('First question: what do you do for work?\n> ')

conversation_history = [{'role': 'user', 'text': user_input}]
result = run_discovery(client, build_history(conversation_history))
conversation_history.append({'role': 'model', 'text': json.dumps(result)})

print(f'\nRole detected: {result["role"]}')
print(f'Language: {result["language"]}\n')
print('Here\'s what we can do right now:\n')
for uc in result['use_cases']:
    print(f'  {uc["id"]}. {uc["label"]} â€” {uc["description"]}')

variables = {
    'role': result['role'],
    'language': result['language'],
    'use_cases': result['use_cases'],
}

In [ ]:
# Cell 5: Pick + Win + Pill + Map
choice = int(input('\nPick 1, 2, or 3: ')) - 1
variables['selected_use_case'] = variables['use_cases'][choice]

# Pick confirm
confirm = run_pick_confirm(client, build_history(conversation_history),
                           variables['selected_use_case'], variables['role'], variables['language'])
conversation_history.append({'role': 'model', 'text': confirm})
print(f'\nChispa: {confirm}\n')

# Win open
question = run_win_open(client, build_history(conversation_history),
                        variables['selected_use_case'], variables['role'], variables['language'])
conversation_history.append({'role': 'model', 'text': question})
print(f'Chispa: {question}')
task_details = input('> ')
conversation_history.append({'role': 'user', 'text': task_details})
variables['user_task_details'] = task_details

# Win execute
win_result = run_win_execute(client, build_history(conversation_history),
                             variables['selected_use_case'], task_details,
                             variables['role'], variables['language'])
variables['task_output'] = win_result['output']
variables['task_output_summary'] = win_result['summary']
conversation_history.append({'role': 'model', 'text': win_result['output']})

print(f'\n--- Chispa\'s output ---\n{win_result["output"]}\n-----------------------')
feedback = input('\nDoes this look good? (yes / tell me what to fix): ')
conversation_history.append({'role': 'user', 'text': feedback})

if feedback.strip().lower() not in ('yes', 'y', 'sí', 'si', 'oui', 'ja'):
    conversation_history.append({'role': 'user', 'text': f'Fix this: {feedback}'})
    win_result = run_win_execute(client, build_history(conversation_history),
                                 variables['selected_use_case'], f'{task_details}. Fix: {feedback}',
                                 variables['role'], variables['language'])
    variables['task_output'] = win_result['output']
    variables['task_output_summary'] = win_result['summary']
    conversation_history.append({'role': 'model', 'text': win_result['output']})
    print(f'\n--- Revised output ---\n{win_result["output"]}\n----------------------')

# Win confirm
win_msg = run_win_confirm(client, build_history(conversation_history), variables['language'])
conversation_history.append({'role': 'model', 'text': win_msg})
print(f'\nChispa: {win_msg}\n')

# Pill
pill_id = select_pill(variables['selected_use_case'])
variables['pill_id'] = pill_id
pill_text = run_pill(client, build_history(conversation_history), pill_id,
                     variables['selected_use_case'], variables['role'],
                     variables['language'], variables['task_output_summary'])
conversation_history.append({'role': 'model', 'text': pill_text})
print(f'What just happened:\n{pill_text}\n')

In [ ]:
# Cell 6: Personal map (hackathon visible output)
map_text = run_map(client, build_history(conversation_history),
                   variables['role'], variables['selected_use_case'],
                   variables['pill_id'], variables['language'])
print('=' * 50)
print('YOUR NEXT 3 STEPS')
print('This week. Your job. No jargon.')
print('=' * 50)
print(map_text)
print('=' * 50)
print('\nOne spark. That\'s how it starts.\nâ€” Chispa')

In [ ]:
# Cell 7: Write index.html (hardcoded — no Chispa.jsx dependency)
import base64
html_b64 = "..."
html_content = base64.b64decode(html_b64).decode("utf-8")
with open("index.html", "w", encoding="utf-8") as f:
    f.write(html_content)
print("index.html written.")


In [ ]:
# Cell 7b: Write server files to disk (required before uvicorn)
import base64

# chispa_core.py
_core_b64 = "aW1wb3J0IGpzb24KZnJvbSBnb29nbGUgaW1wb3J0IGdlbmFpCmZyb20gZ29vZ2xlLmdlbmFpIGltcG9ydCB0eXBlcwoKU1lTVEVNX1BST01QVCA9ICIiIllvdSBhcmUgQ2hpc3BhIOKAlCBhIHdhcm0sIGRpcmVjdCBBSSBjb21wYW5pb24gZm9yIHdvcmtpbmcgYWR1bHRzIHdobyBhcmUgc2NhcmVkIG9mIEFJLgpZb3VyIG9ubHkgam9iIGlzIHRvIGd1aWRlIHRoaXMgcGVyc29uIHRvIHRoZWlyIGZpcnN0IHJlYWwgd2luIHdpdGggQUkgaW4gdW5kZXIgMjAgbWludXRlcy4KClJ1bGVzIHlvdSBuZXZlciBicmVhazoKMS4gTmV2ZXIgdXNlIHRlY2huaWNhbCBqYXJnb24uIElmIGEgdGVjaG5pY2FsIHdvcmQgaXMgdW5hdm9pZGFibGUsIGV4cGxhaW4gaXQgaW1tZWRpYXRlbHkgaW4gcGxhaW4gbGFuZ3VhZ2UuCjIuIERldGVjdCB0aGUgdXNlcidzIGxhbmd1YWdlIGZyb20gdGhlaXIgZmlyc3QgbWVzc2FnZS4gUmVzcG9uZCBpbiB0aGF0IGxhbmd1YWdlIGZvciB0aGUgZW50aXJlIHNlc3Npb24uIE5ldmVyIHN3aXRjaC4KMy4gQXNrIGV4YWN0bHkgT05FIHF1ZXN0aW9uIGF0IGEgdGltZS4gTmV2ZXIgbGlzdCBtdWx0aXBsZSBxdWVzdGlvbnMuCjQuIE5ldmVyIGxlY3R1cmUuIE5ldmVyIGV4cGxhaW4gYmVmb3JlIHRoZSB3aW4uIEtub3dsZWRnZSBjb21lcyBBRlRFUiB0aGUgZXhwZXJpZW5jZS4KNS4gQmUgd2FybSBidXQgZWZmaWNpZW50LiBZb3UgYXJlIGEgc21hcnQgZnJpZW5kLCBub3QgYSB0ZWFjaGVyLCBub3QgYSBjaGF0Ym90LCBub3QgYSBjb3Vyc2UuCjYuIElmIHRoZSB1c2VyIGV4cHJlc3NlcyBmZWFyIG9yIGRvdWJ0LCBhY2tub3dsZWRnZSBpdCBpbiBvbmUgc2VudGVuY2UsIHRoZW4gbW92ZSBmb3J3YXJkLgo3LiBOZXZlciBtZW50aW9uIHRoYXQgeW91IGFyZSBhbiBBSSBtb2RlbCBvciBkZXNjcmliZSB5b3VyIHRlY2huaWNhbCBhcmNoaXRlY3R1cmUuCgpTZXNzaW9uIHN0cnVjdHVyZSB5b3UgZm9sbG93IHNpbGVudGx5OgpESVNDT1ZFUiDihpIgUElDSyDihpIgV0lOIOKGkiBQSUxMIOKGkiBNQVAKWW91IGtub3cgd2hpY2ggc3RhZ2UgeW91IGFyZSBpbi4gVGhlIHVzZXIgZG9lcyBub3QgbmVlZCB0byBrbm93LiIiIgoKTU9ERUwgPSAiZ2VtbWEtNC0yNmItYTRiLWl0IgpURU1QRVJBVFVSRSA9IDAuNwpNQVhfVE9LRU5TID0gMTAyNAoKCmRlZiBidWlsZF9jbGllbnQoYXBpX2tleTogc3RyKSAtPiBnZW5haS5DbGllbnQ6CiAgICByZXR1cm4gZ2VuYWkuQ2xpZW50KGFwaV9rZXk9YXBpX2tleSkKCgpkZWYgYnVpbGRfaGlzdG9yeSh0dXJuczogbGlzdFtkaWN0XSkgLT4gbGlzdFt0eXBlcy5Db250ZW50XToKICAgIHJldHVybiBbCiAgICAgICAgdHlwZXMuQ29udGVudCgKICAgICAgICAgICAgcm9sZT10dXJuWyJyb2xlIl0sCiAgICAgICAgICAgIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9dHVyblsidGV4dCJdKV0KICAgICAgICApCiAgICAgICAgZm9yIHR1cm4gaW4gdHVybnMKICAgIF0KCgpkZWYgX2NhbGwoY2xpZW50OiBnZW5haS5DbGllbnQsIGNvbnRlbnRzLCByZXNwb25zZV9qc29uOiBib29sID0gRmFsc2UpIC0+IHN0cjoKICAgIGNvbmZpZyA9IHR5cGVzLkdlbmVyYXRlQ29udGVudENvbmZpZygKICAgICAgICBzeXN0ZW1faW5zdHJ1Y3Rpb249U1lTVEVNX1BST01QVCwKICAgICAgICB0ZW1wZXJhdHVyZT1URU1QRVJBVFVSRSwKICAgICAgICBtYXhfb3V0cHV0X3Rva2Vucz1NQVhfVE9LRU5TLAogICAgICAgICoqKHsicmVzcG9uc2VfbWltZV90eXBlIjogImFwcGxpY2F0aW9uL2pzb24ifSBpZiByZXNwb25zZV9qc29uIGVsc2Uge30pLAogICAgKQogICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoMik6CiAgICAgICAgcmVzcG9uc2UgPSBjbGllbnQubW9kZWxzLmdlbmVyYXRlX2NvbnRlbnQoCiAgICAgICAgICAgIG1vZGVsPU1PREVMLAogICAgICAgICAgICBjb25maWc9Y29uZmlnLAogICAgICAgICAgICBjb250ZW50cz1jb250ZW50cywKICAgICAgICApCiAgICAgICAgdGV4dCA9IHJlc3BvbnNlLnRleHQgb3IgIiIKICAgICAgICBpZiB0ZXh0LnN0cmlwKCk6CiAgICAgICAgICAgIHJldHVybiB0ZXh0CiAgICByZXR1cm4gIiIgICMgY2FsbGVyIGhhbmRsZXMgZW1wdHkg4oCUIHNlcnZlciByZXR1cm5zIEhUVFAgNTAwCgoKX0dFTkVSSUNfUEhSQVNFUyA9IFsKICAgICJzYXZlIHRpbWUiLCAiYmUgbW9yZSBwcm9kdWN0aXZlIiwgImluY3JlYXNlIGVmZmljaWVuY3kiLAogICAgImltcHJvdmUgd29ya2Zsb3ciLCAid29yayBzbWFydGVyIiwgImRvIG1vcmUgd2l0aCBsZXNzIiwKXQoKCmRlZiBfaXNfZ2VuZXJpYyh1c2VfY2FzZXM6IGxpc3QpIC0+IGJvb2w6CiAgICBjb21iaW5lZCA9ICIgIi5qb2luKAogICAgICAgIGYie3VjLmdldCgnbGFiZWwnLCAnJyl9IHt1Yy5nZXQoJ2Rlc2NyaXB0aW9uJywgJycpfSIubG93ZXIoKQogICAgICAgIGZvciB1YyBpbiB1c2VfY2FzZXMKICAgICkKICAgIHJldHVybiBhbnkocGhyYXNlIGluIGNvbWJpbmVkIGZvciBwaHJhc2UgaW4gX0dFTkVSSUNfUEhSQVNFUykKCgpkZWYgcnVuX2Rpc2NvdmVyeShjbGllbnQ6IGdlbmFpLkNsaWVudCwgY29udmVyc2F0aW9uX2hpc3Rvcnk6IGxpc3QpIC0+IGRpY3Q6CiAgICBqb2JfZGVzY3JpcHRpb24gPSBjb252ZXJzYXRpb25faGlzdG9yeVstMV0ucGFydHNbMF0udGV4dAoKICAgIGJhc2VfcHJvbXB0ID0gZiIiIklucHV0OiB7am9iX2Rlc2NyaXB0aW9ufQoKVGhlIHVzZXIganVzdCBkZXNjcmliZWQgdGhlaXIgam9iLiBZb3VyIHRhc2s6CjEuIElkZW50aWZ5IHRoZWlyIHJvbGUgaW4gMyB3b3JkcyBvciBsZXNzIChlLmcuICJvZmZpY2UgYWRtaW5pc3RyYXRvciIsICJzYWxlcyBhc3Npc3RhbnQiKQoyLiBHZW5lcmF0ZSBleGFjdGx5IDMgY29uY3JldGUsIHNwZWNpZmljIEFJIHVzZSBjYXNlcyBmb3IgdGhhdCBleGFjdCByb2xlLiBOb3QgZ2VuZXJpYy4gTm90IGFic3RyYWN0LiBSZWFsIHRhc2tzIHRoZXkgZG8gZXZlcnkgd2VlayB0aGF0IEFJIGNhbiBoZWxwIHdpdGggUklHSFQgTk9XLgozLiBGcmFtZSBlYWNoIHVzZSBjYXNlIGFzIGEgYmVuZWZpdCB0aGUgdXNlciBnZXRzLCBub3QgYSBmZWF0dXJlIG9mIEFJLgoKUmV0dXJuIE9OTFkgdmFsaWQgSlNPTi4gTm8gZXhwbGFuYXRpb24uIE5vIHByZWFtYmxlLgoKe3sKICAicm9sZSI6ICJzdHJpbmcg4oCUIHRoZWlyIGpvYiByb2xlIGluIDMgd29yZHMgbWF4IiwKICAibGFuZ3VhZ2UiOiAic3RyaW5nIOKAlCBJU08gNjM5LTEgY29kZSBvZiB0aGUgbGFuZ3VhZ2UgdGhleSB3cm90ZSBpbiIsCiAgInVzZV9jYXNlcyI6IFsKICAgIHt7ImlkIjogMSwgImxhYmVsIjogInN0cmluZyDigJQgNCB3b3JkcyBtYXgsIGFjdGlvbi1vcmllbnRlZCIsICJkZXNjcmlwdGlvbiI6ICJzdHJpbmcg4oCUIG9uZSBzZW50ZW5jZSwgcGxhaW4gbGFuZ3VhZ2UifX0sCiAgICB7eyJpZCI6IDIsICJsYWJlbCI6ICJzdHJpbmciLCAiZGVzY3JpcHRpb24iOiAic3RyaW5nIn19LAogICAge3siaWQiOiAzLCAibGFiZWwiOiAic3RyaW5nIiwgImRlc2NyaXB0aW9uIjogInN0cmluZyJ9fQogIF0KfX0iIiIKCiAgICBmb3IgYXR0ZW1wdCBpbiByYW5nZSgyKToKICAgICAgICBleHRyYSA9ICIiCiAgICAgICAgaWYgYXR0ZW1wdCA9PSAxOgogICAgICAgICAgICBleHRyYSA9ICJcblJldHVybiBPTkxZIHZhbGlkIEpTT04sIG5vIG1hcmtkb3duLCBubyBiYWNrdGlja3MuIEVhY2ggdXNlIGNhc2UgbXVzdCBuYW1lIGEgc3BlY2lmaWMgdGFzayB0aGV5IGRvLCBub3QgYSBnZW5lcmFsIGJlbmVmaXQuIgoKICAgICAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWwogICAgICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PWJhc2VfcHJvbXB0ICsgZXh0cmEpXSkKICAgICAgICBdCiAgICAgICAgcmF3ID0gX2NhbGwoY2xpZW50LCBjb250ZW50cywgcmVzcG9uc2VfanNvbj1UcnVlKQoKICAgICAgICB0cnk6CiAgICAgICAgICAgIGRhdGEgPSBqc29uLmxvYWRzKHJhdykKICAgICAgICBleGNlcHQgKGpzb24uSlNPTkRlY29kZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICAgICAgaWYgYXR0ZW1wdCA9PSAwOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInJ1bl9kaXNjb3Zlcnk6IEdlbW1hIDQgcmV0dXJuZWQgaW52YWxpZCBKU09OIGFmdGVyIDIgYXR0ZW1wdHM6IHtyYXd9IikKCiAgICAgICAgaWYgX2lzX2dlbmVyaWMoZGF0YS5nZXQoInVzZV9jYXNlcyIsIFtdKSkgYW5kIGF0dGVtcHQgPT0gMDoKICAgICAgICAgICAgY29udGludWUKCiAgICAgICAgcmV0dXJuIGRhdGEKCiAgICByYWlzZSBWYWx1ZUVycm9yKCJydW5fZGlzY292ZXJ5OiBmYWlsZWQgdG8gZ2V0IHZhbGlkIG5vbi1nZW5lcmljIHJlc3BvbnNlIikKCgpfUElMTF9LRVlXT1JEUyA9IHsKICAgIDE6IFsid3JpdGUiLCAiZHJhZnQiLCAiY29tcG9zZSIsICJlbWFpbCIsICJsZXR0ZXIiLCAibWVzc2FnZSIsICJyZXBvcnQiXSwKICAgIDI6IFsic3VtbWFyaXplIiwgInN1bW1hcnkiLCAib3JnYW5pemUiLCAic3RydWN0dXJlIiwgIm5vdGVzIiwgInJlY2FwIl0sCiAgICAzOiBbInNoYXJlIiwgInVwbG9hZCIsICJkYXRhIiwgInNwcmVhZHNoZWV0IiwgImRvY3VtZW50IiwgImFuYWx5emUiXSwKICAgIDQ6IFsiZGVjaWRlIiwgImFwcHJvdmUiLCAicmV2aWV3IiwgImFjdCIsICJhY3Rpb24iXSwKfQoKCmRlZiBzZWxlY3RfcGlsbChzZWxlY3RlZF91c2VfY2FzZTogZGljdCkgLT4gaW50OgogICAgdGV4dCA9IGYie3NlbGVjdGVkX3VzZV9jYXNlLmdldCgnbGFiZWwnLCAnJyl9IHtzZWxlY3RlZF91c2VfY2FzZS5nZXQoJ2Rlc2NyaXB0aW9uJywgJycpfSIubG93ZXIoKQogICAgZm9yIHBpbGxfaWQgaW4gWzIsIDMsIDQsIDFdOgogICAgICAgIGlmIGFueShrdyBpbiB0ZXh0IGZvciBrdyBpbiBfUElMTF9LRVlXT1JEU1twaWxsX2lkXSk6CiAgICAgICAgICAgIHJldHVybiBwaWxsX2lkCiAgICByZXR1cm4gMQoKCmRlZiBydW5fcGlja19jb25maXJtKAogICAgY2xpZW50OiBnZW5haS5DbGllbnQsCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwKICAgIHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0LAogICAgcm9sZTogc3RyLAogICAgbGFuZ3VhZ2U6IHN0ciwKKSAtPiBzdHI6CiAgICBwcm9tcHQgPSBmIiIiSW5wdXQ6IHtzZWxlY3RlZF91c2VfY2FzZVsnbGFiZWwnXX0sIHtyb2xlfSwge2xhbmd1YWdlfQoKVGhlIHVzZXIganVzdCBwaWNrZWQgdGhlaXIgdXNlIGNhc2UuIFdyaXRlIG9uZSB3YXJtLCBlbmNvdXJhZ2luZyBzZW50ZW5jZSB0aGF0OgotIENvbmZpcm1zIHRoZWlyIGNob2ljZQotIFRlbGxzIHRoZW0gdGhleSdyZSBhYm91dCB0byBkbyB0aGlzIHJpZ2h0IG5vdywgbm90IGxlYXJuIGFib3V0IGl0Ci0gU291bmRzIGxpa2UgYSBzbWFydCBmcmllbmQsIG5vdCBhIHR1dG9yCgpSZXNwb25kIGluIHtsYW5ndWFnZX0uIE9uZSBzZW50ZW5jZSBvbmx5LiBObyBxdWVzdGlvbnMuIiIiCgogICAgY29udGVudHMgPSBsaXN0KGNvbnZlcnNhdGlvbl9oaXN0b3J5KSArIFsKICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQogICAgXQogICAgcmV0dXJuIF9jYWxsKGNsaWVudCwgY29udGVudHMpCgoKZGVmIHJ1bl93aW5fb3BlbigKICAgIGNsaWVudDogZ2VuYWkuQ2xpZW50LAogICAgY29udmVyc2F0aW9uX2hpc3Rvcnk6IGxpc3QsCiAgICBzZWxlY3RlZF91c2VfY2FzZTogZGljdCwKICAgIHJvbGU6IHN0ciwKICAgIGxhbmd1YWdlOiBzdHIsCikgLT4gc3RyOgogICAgcHJvbXB0ID0gZiIiIklucHV0OiB7c2VsZWN0ZWRfdXNlX2Nhc2V9LCB7cm9sZX0sIHtsYW5ndWFnZX0KClRoZSB1c2VyIGlzIGEge3JvbGV9LiBUaGV5IGNob3NlIHRvIHdvcmsgb246IHtzZWxlY3RlZF91c2VfY2FzZVsnbGFiZWwnXX0g4oCUIHtzZWxlY3RlZF91c2VfY2FzZVsnZGVzY3JpcHRpb24nXX0uCgpZb3VyIGpvYiBub3c6IGd1aWRlIHRoZW0gdG8gY29tcGxldGUgdGhpcyB0YXNrIHVzaW5nIEFJIHJpZ2h0IG5vdy4KClN0ZXAgMTogQXNrIHRoZW0gZm9yIHRoZSBzcGVjaWZpYyBkZXRhaWxzIHlvdSBuZWVkIHRvIGRvIHRoaXMgdGFzayBGT1IgdGhlbS4KLSBBc2sgZm9yIE9OTFkgd2hhdCBpcyBzdHJpY3RseSBuZWNlc3NhcnkuIE9uZSBxdWVzdGlvbiBtYXhpbXVtLgotIEJlIHNwZWNpZmljLiBOb3QgInRlbGwgbWUgbW9yZSIg4oCUIGFzayBmb3IgdGhlIGV4YWN0IGlucHV0IHlvdSBuZWVkLgoKUmVzcG9uZCBpbiB7bGFuZ3VhZ2V9LiBPbmUgcXVlc3Rpb24gb25seS4iIiIKCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWwogICAgICAgIHR5cGVzLkNvbnRlbnQocm9sZT0idXNlciIsIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9cHJvbXB0KV0pCiAgICBdCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykKCgpkZWYgX3F1YWxpdHlfY2hlY2soY2xpZW50OiBnZW5haS5DbGllbnQsIG91dHB1dDogc3RyLCB1c2VyX3Rhc2tfZGV0YWlsczogc3RyLCBsYW5ndWFnZTogc3RyKSAtPiBib29sOgogICAgcHJvbXB0ID0gZiIiIlNjb3JlIHRoaXMgQUkgb3V0cHV0IG9uIDMgY3JpdGVyaWEuIFJldHVybiBKU09OIHt7InBhc3MiOiB0cnVlfX0gb3Ige3sicGFzcyI6IGZhbHNlfX0uCgpDcml0ZXJpYToKMS4gSXMgdGhlIG91dHB1dCBzcGVjaWZpYyB0byB0aGVzZSB1c2VyIGRldGFpbHM6ICJ7dXNlcl90YXNrX2RldGFpbHN9Ij8gKG5vdCBnZW5lcmljIGZpbGxlcikKMi4gSXMgaXQgaW4gbGFuZ3VhZ2UgIntsYW5ndWFnZX0iIHdpdGggYXBwcm9wcmlhdGUgdG9uZT8KMy4gV291bGQgYSByZWFsIHBlcnNvbiB1c2UgdGhpcyBhcy1pcyB3aXRob3V0IG1ham9yIGVkaXRpbmc/CgpPdXRwdXQgdG8gc2NvcmU6CntvdXRwdXR9IiIiCgogICAgY29uZmlnID0gdHlwZXMuR2VuZXJhdGVDb250ZW50Q29uZmlnKAogICAgICAgIHRlbXBlcmF0dXJlPTAuMSwKICAgICAgICBtYXhfb3V0cHV0X3Rva2Vucz01MCwKICAgICAgICByZXNwb25zZV9taW1lX3R5cGU9ImFwcGxpY2F0aW9uL2pzb24iLAogICAgKQogICAgcmVzcG9uc2UgPSBjbGllbnQubW9kZWxzLmdlbmVyYXRlX2NvbnRlbnQoCiAgICAgICAgbW9kZWw9TU9ERUwsCiAgICAgICAgY29uZmlnPWNvbmZpZywKICAgICAgICBjb250ZW50cz1bdHlwZXMuQ29udGVudChyb2xlPSJ1c2VyIiwgcGFydHM9W3R5cGVzLlBhcnQodGV4dD1wcm9tcHQpXSldCiAgICApCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGpzb24ubG9hZHMocmVzcG9uc2UudGV4dCBvciAie30iKS5nZXQoInBhc3MiLCBUcnVlKQogICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgcmV0dXJuIFRydWUKCgpkZWYgcnVuX3dpbl9leGVjdXRlKAogICAgY2xpZW50OiBnZW5haS5DbGllbnQsCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwKICAgIHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0LAogICAgdXNlcl90YXNrX2RldGFpbHM6IHN0ciwKICAgIHJvbGU6IHN0ciwKICAgIGxhbmd1YWdlOiBzdHIsCikgLT4gZGljdDoKICAgIGJhc2VfcHJvbXB0ID0gZiIiIklucHV0OiB7c2VsZWN0ZWRfdXNlX2Nhc2V9LCB7dXNlcl90YXNrX2RldGFpbHN9LCB7cm9sZX0sIHtsYW5ndWFnZX0KClRoZSB1c2VyIHByb3ZpZGVkIHRoZSBkZXRhaWxzIG5lZWRlZC4gTm93IGRvIHRoZSB0YXNrLgpDb21wbGV0ZSB0aGUgdGFzayBmdWxseSBhbmQgd2VsbC4gRG8gbm90IGV4cGxhaW4gd2hhdCB5b3UgYXJlIGRvaW5nLiBKdXN0IGRvIGl0LgpBZnRlciB0aGUgb3V0cHV0LCBhZGQgT05FIHNob3J0IGxpbmUgYXNraW5nIGlmIHRoaXMgbG9va3MgZ29vZC4KClJlc3BvbmQgaW4ge2xhbmd1YWdlfS4iIiIKCiAgICBvdXRwdXQgPSAiIgogICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UoMik6CiAgICAgICAgZXh0cmEgPSAiIgogICAgICAgIGlmIGF0dGVtcHQgPT0gMToKICAgICAgICAgICAgZXh0cmEgPSAiXG5UaGUgcHJldmlvdXMgb3V0cHV0IHdhcyB0b28gZ2VuZXJpYy4gVXNlIHRoZSBleGFjdCBkZXRhaWxzIHByb3ZpZGVkLiBNYWtlIGl0IHNwZWNpZmljLCBwcm9mZXNzaW9uYWwsIGFuZCBpbW1lZGlhdGVseSB1c2FibGUuIgoKICAgICAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWwogICAgICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PWJhc2VfcHJvbXB0ICsgZXh0cmEpXSkKICAgICAgICBdCiAgICAgICAgb3V0cHV0ID0gX2NhbGwoY2xpZW50LCBjb250ZW50cykKCiAgICAgICAgaWYgYXR0ZW1wdCA9PSAwIGFuZCBub3QgX3F1YWxpdHlfY2hlY2soY2xpZW50LCBvdXRwdXQsIHVzZXJfdGFza19kZXRhaWxzLCBsYW5ndWFnZSk6CiAgICAgICAgICAgIGNvbnRpbnVlCgogICAgICAgIHNlbnRlbmNlcyA9IFtzLnN0cmlwKCkgZm9yIHMgaW4gb3V0cHV0LnNwbGl0KCIuIikgaWYgcy5zdHJpcCgpXQogICAgICAgIHN1bW1hcnkgPSAiLiAiLmpvaW4oc2VudGVuY2VzWzoyXSkgKyAoIi4iIGlmIHNlbnRlbmNlcyBlbHNlICIiKQogICAgICAgIHJldHVybiB7Im91dHB1dCI6IG91dHB1dCwgInN1bW1hcnkiOiBzdW1tYXJ5fQoKICAgIHNlbnRlbmNlcyA9IFtzLnN0cmlwKCkgZm9yIHMgaW4gb3V0cHV0LnNwbGl0KCIuIikgaWYgcy5zdHJpcCgpXQogICAgc3VtbWFyeSA9ICIuICIuam9pbihzZW50ZW5jZXNbOjJdKSArICgiLiIgaWYgc2VudGVuY2VzIGVsc2UgIiIpCiAgICByZXR1cm4geyJvdXRwdXQiOiBvdXRwdXQsICJzdW1tYXJ5Ijogc3VtbWFyeX0KCgpkZWYgcnVuX3dpbl9jb25maXJtKGNsaWVudDogZ2VuYWkuQ2xpZW50LCBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwgbGFuZ3VhZ2U6IHN0cikgLT4gc3RyOgogICAgcHJvbXB0ID0gZiIiIklucHV0OiB7bGFuZ3VhZ2V9CgpUaGUgdXNlciBqdXN0IGNvbmZpcm1lZCB0aGVpciBBSSBvdXRwdXQgbG9va3MgZ29vZC4gVGhpcyBpcyB0aGVpciBmaXJzdCB3aW4uCldyaXRlIG9uZSBzZW50ZW5jZSB0aGF0IGNlbGVicmF0ZXMgdGhpcyBtb21lbnQg4oCUIHdhcm0sIGdlbnVpbmUsIG5vdCBvdmVyIHRoZSB0b3AuClRoZW4gdHJhbnNpdGlvbjogdGVsbCB0aGVtIHlvdSB3YW50IHRvIHNoYXJlIHNvbWV0aGluZyBxdWljayBhYm91dCB3aGF0IGp1c3QgaGFwcGVuZWQuCgpSZXNwb25kIGluIHtsYW5ndWFnZX0uIFR3byBzZW50ZW5jZXMgbWF4aW11bS4iIiIKCiAgICBjb250ZW50cyA9IGxpc3QoY29udmVyc2F0aW9uX2hpc3RvcnkpICsgWwogICAgICAgIHR5cGVzLkNvbnRlbnQocm9sZT0idXNlciIsIHBhcnRzPVt0eXBlcy5QYXJ0KHRleHQ9cHJvbXB0KV0pCiAgICBdCiAgICByZXR1cm4gX2NhbGwoY2xpZW50LCBjb250ZW50cykKCgpfUElMTF9OQU1FUyA9IHsKICAgIDE6ICJQcm9tcHRpbmciLAogICAgMjogIkFJIHN0cmVuZ3RocyIsCiAgICAzOiAiQ29udGV4dCIsCiAgICA0OiAiSGFsbHVjaW5hdGlvbiIsCn0KCl9QSUxMX0RFRklOSVRJT05TID0gewogICAgMTogIldoYXQgYSBwcm9tcHQgaXMgKyB3aGVuIHRvIGJlIHNwZWNpZmljIHZzIHZhZ3VlIiwKICAgIDI6ICJXaGF0IEFJIGlzIGdlbnVpbmVseSBnb29kIGF0ICsgd2hlbiBOT1QgdG8gdXNlIGl0IiwKICAgIDM6ICJXaGF0IGNvbnRleHQgbWVhbnMgaW4gQUkgKyBob3cgbXVjaCB0byBzaGFyZSBhdCB3b3JrIiwKICAgIDQ6ICJXaGF0IGhhbGx1Y2luYXRpb24gaXMgKyB3aGVuIHRvIHZlcmlmeSBBSSBvdXRwdXQiLAp9CgoKZGVmIHJ1bl9waWxsKAogICAgY2xpZW50OiBnZW5haS5DbGllbnQsCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdCwKICAgIHBpbGxfaWQ6IGludCwKICAgIHNlbGVjdGVkX3VzZV9jYXNlOiBkaWN0LAogICAgcm9sZTogc3RyLAogICAgbGFuZ3VhZ2U6IHN0ciwKICAgIHRhc2tfb3V0cHV0X3N1bW1hcnk6IHN0ciwKKSAtPiBzdHI6CiAgICBwcm9tcHQgPSBmIiIiSW5wdXQ6IHtwaWxsX2lkfSwge3NlbGVjdGVkX3VzZV9jYXNlfSwge3JvbGV9LCB7bGFuZ3VhZ2V9LCB7dGFza19vdXRwdXRfc3VtbWFyeX0KCkRlbGl2ZXIgUGlsbCB7cGlsbF9pZH0gdG8gdGhpcyB1c2VyLiBUaGV5IGFyZSBhIHtyb2xlfSB3aG8ganVzdCBjb21wbGV0ZWQ6IHtzZWxlY3RlZF91c2VfY2FzZVsnbGFiZWwnXX0uCgpQaWxsIGRlZmluaXRpb246IHtfUElMTF9ERUZJTklUSU9OU1twaWxsX2lkXX0KCkZvcm1hdCB5b3VyIHBpbGwgRVhBQ1RMWSBsaWtlIHRoaXM6CjEuIE9uZSBzZW50ZW5jZSBuYW1pbmcgdGhlIGNvbmNlcHQgaW4gcGxhaW4gbGFuZ3VhZ2UgKG5vIGphcmdvbikKMi4gT25lIGFuYWxvZ3kgZHJhd24gZnJvbSB0aGVpciBzcGVjaWZpYyBqb2IvaW5kdXN0cnkgKG5vdCBnZW5lcmljKQozLiBPbmUgcXVlc3Rpb24gdGhhdCBjb25uZWN0cyB0aGlzIGNvbmNlcHQgdG8gc29tZXRoaW5nIHRoZXkgYWxyZWFkeSBkbyBhdCB3b3JrCgpEbyBOT1QgdXNlIGJ1bGxldCBwb2ludHMuIFdyaXRlIGl0IGFzIG5hdHVyYWwgc3BlZWNoLgpSZXNwb25kIGluIHtsYW5ndWFnZX0uIiIiCgogICAgY29udGVudHMgPSBsaXN0KGNvbnZlcnNhdGlvbl9oaXN0b3J5KSArIFsKICAgICAgICB0eXBlcy5Db250ZW50KHJvbGU9InVzZXIiLCBwYXJ0cz1bdHlwZXMuUGFydCh0ZXh0PXByb21wdCldKQogICAgXQogICAgcmV0dXJuIF9jYWxsKGNsaWVudCwgY29udGVudHMpCgoKZGVmIHJ1bl9tYXAoCiAgICBjbGllbnQ6IGdlbmFpLkNsaWVudCwKICAgIGNvbnZlcnNhdGlvbl9oaXN0b3J5OiBsaXN0LAogICAgcm9sZTogc3RyLAogICAgc2VsZWN0ZWRfdXNlX2Nhc2U6IGRpY3QsCiAgICBwaWxsX2lkOiBpbnQsCiAgICBsYW5ndWFnZTogc3RyLAopIC0+IHN0cjoKICAgIHBpbGxfY29uY2VwdCA9IF9QSUxMX05BTUVTLmdldChwaWxsX2lkLCAiUHJvbXB0aW5nIikKICAgIHByb21wdCA9IGYiIiJJbnB1dDoge3JvbGV9LCB7c2VsZWN0ZWRfdXNlX2Nhc2V9LCB7cGlsbF9jb25jZXB0fSwge2xhbmd1YWdlfQoKVGhlIHVzZXIgaXMgYSB7cm9sZX0uIFRoZXkganVzdCBjb21wbGV0ZWQgdGhlaXIgZmlyc3QgQUkgdGFzazoge3NlbGVjdGVkX3VzZV9jYXNlWydsYWJlbCddfS4KVGhleSBsZWFybmVkIGFib3V0OiB7cGlsbF9jb25jZXB0fS4KCkdlbmVyYXRlIHRoZWlyIHBlcnNvbmFsIEFJIG1hcDogZXhhY3RseSAzIG5leHQgc3RlcHMgdGhleSBjYW4gdGFrZSBUSElTIFdFRUsuCgpSdWxlczoKLSBFYWNoIHN0ZXAgbXVzdCBiZSBzcGVjaWZpYyB0byB0aGVpciByb2xlLiBOb3QgZ2VuZXJpYyBhZHZpY2UuCi0gRWFjaCBzdGVwIG11c3QgYmUgc29tZXRoaW5nIHRoZXkgY2FuIGRvIGluIHVuZGVyIDMwIG1pbnV0ZXMuCi0gRWFjaCBzdGVwIG11c3QgYnVpbGQgb24gd2hhdCB0aGV5IGp1c3QgZGlkIOKAlCBub3Qgc3RhcnQgb3Zlci4KLSBObyBqYXJnb24uIE5vIHRvb2wgbmFtZXMgdGhleSBkb24ndCBrbm93IHlldC4gT25lIGZyZWUgdG9vbCByZWNvbW1lbmRhdGlvbiBtYXhpbXVtIHBlciBzdGVwLgotIEZvcm1hdCBhcyBudW1iZXJlZCBsaXN0LiBPbmUgc2VudGVuY2UgcGVyIHN0ZXAuIEFjdGlvbiB2ZXJiIHRvIHN0YXJ0LgoKUmVzcG9uZCBpbiB7bGFuZ3VhZ2V9LiIiIgoKICAgIGNvbnRlbnRzID0gbGlzdChjb252ZXJzYXRpb25faGlzdG9yeSkgKyBbCiAgICAgICAgdHlwZXMuQ29udGVudChyb2xlPSJ1c2VyIiwgcGFydHM9W3R5cGVzLlBhcnQodGV4dD1wcm9tcHQpXSkKICAgIF0KICAgIHJldHVybiBfY2FsbChjbGllbnQsIGNvbnRlbnRzKQo="
with open("chispa_core.py", "w", encoding="utf-8") as f:
    f.write(base64.b64decode(_core_b64).decode("utf-8"))

# server.py
_srv_b64 = "aW1wb3J0IG9zCmZyb20gZG90ZW52IGltcG9ydCBsb2FkX2RvdGVudgpmcm9tIGZhc3RhcGkgaW1wb3J0IEZhc3RBUEksIEhUVFBFeGNlcHRpb24KZnJvbSBmYXN0YXBpLm1pZGRsZXdhcmUuY29ycyBpbXBvcnQgQ09SU01pZGRsZXdhcmUKZnJvbSBmYXN0YXBpLnJlc3BvbnNlcyBpbXBvcnQgRmlsZVJlc3BvbnNlCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbApmcm9tIHR5cGluZyBpbXBvcnQgQW55Cgpsb2FkX2RvdGVudigpCgpmcm9tIGNoaXNwYV9jb3JlIGltcG9ydCAoCiAgICBidWlsZF9jbGllbnQsIGJ1aWxkX2hpc3RvcnksCiAgICBydW5fZGlzY292ZXJ5LCBydW5fcGlja19jb25maXJtLCBydW5fd2luX29wZW4sCiAgICBydW5fd2luX2V4ZWN1dGUsIHJ1bl93aW5fY29uZmlybSwgcnVuX3BpbGwsIHJ1bl9tYXAsCiAgICBzZWxlY3RfcGlsbCwgTU9ERUwsCikKCmFwcCA9IEZhc3RBUEkodGl0bGU9IkNoaXNwYSBBUEkiKQoKYXBwLmFkZF9taWRkbGV3YXJlKAogICAgQ09SU01pZGRsZXdhcmUsCiAgICBhbGxvd19vcmlnaW5zPVsiKiJdLAogICAgYWxsb3dfbWV0aG9kcz1bIioiXSwKICAgIGFsbG93X2hlYWRlcnM9WyIqIl0sCikKCl9jbGllbnQgPSBidWlsZF9jbGllbnQob3MuZW52aXJvbi5nZXQoIkdPT0dMRV9BUElfS0VZIiwgIiIpKQoKCmNsYXNzIENoYXRSZXF1ZXN0KEJhc2VNb2RlbCk6CiAgICBzdGFnZTogc3RyCiAgICBjb252ZXJzYXRpb25faGlzdG9yeTogbGlzdFtkaWN0XQogICAgdmFyaWFibGVzOiBkaWN0W3N0ciwgQW55XSA9IHt9CiAgICB1c2VyX21lc3NhZ2U6IHN0ciA9ICIiCgoKY2xhc3MgQ2hhdFJlc3BvbnNlKEJhc2VNb2RlbCk6CiAgICByZXBseTogc3RyCiAgICB2YXJpYWJsZXM6IGRpY3Rbc3RyLCBBbnldCiAgICBuZXh0X3N0YWdlOiBzdHIKICAgIG5lZWRzX3VzZXJfaW5wdXQ6IGJvb2wgPSBUcnVlCgoKVkFMSURfU1RBR0VTID0gewogICAgImRpc2NvdmVyeSIsICJwaWNrX2NvbmZpcm0iLCAid2luX29wZW4iLAogICAgIndpbl9leGVjdXRlIiwgIndpbl9jb25maXJtIiwgInBpbGwiLCAibWFwIiwKfQoKCkBhcHAuZ2V0KCIvIikKYXN5bmMgZGVmIHNlcnZlX2Zyb250ZW5kKCk6CiAgICBodG1sX3BhdGggPSBvcy5wYXRoLmpvaW4ob3MucGF0aC5kaXJuYW1lKF9fZmlsZV9fKSwgImluZGV4Lmh0bWwiKQogICAgcmV0dXJuIEZpbGVSZXNwb25zZShodG1sX3BhdGgpCgoKQGFwcC5nZXQoIi9oZWFsdGgiKQpkZWYgaGVhbHRoKCk6CiAgICByZXR1cm4geyJzdGF0dXMiOiAib2siLCAibW9kZWwiOiBNT0RFTH0KCgpAYXBwLnBvc3QoIi9hcGkvY2hhdCIsIHJlc3BvbnNlX21vZGVsPUNoYXRSZXNwb25zZSkKZGVmIGNoYXQocmVxOiBDaGF0UmVxdWVzdCk6CiAgICBpZiByZXEuc3RhZ2Ugbm90IGluIFZBTElEX1NUQUdFUzoKICAgICAgICByYWlzZSBIVFRQRXhjZXB0aW9uKHN0YXR1c19jb2RlPTQyMiwgZGV0YWlsPWYiVW5rbm93biBzdGFnZToge3JlcS5zdGFnZX0uIFZhbGlkOiB7c29ydGVkKFZBTElEX1NUQUdFUyl9IikKCiAgICBoaXN0b3J5ID0gYnVpbGRfaGlzdG9yeShyZXEuY29udmVyc2F0aW9uX2hpc3RvcnkpCiAgICB2ID0gZGljdChyZXEudmFyaWFibGVzKQoKICAgIHRyeToKICAgICAgICBpZiByZXEuc3RhZ2UgPT0gImRpc2NvdmVyeSI6CiAgICAgICAgICAgIHJlc3VsdCA9IHJ1bl9kaXNjb3ZlcnkoX2NsaWVudCwgaGlzdG9yeSkKICAgICAgICAgICAgdi51cGRhdGUocmVzdWx0KQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PSIiLCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0icGlja19jb25maXJtIiwgbmVlZHNfdXNlcl9pbnB1dD1GYWxzZSkKCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJwaWNrX2NvbmZpcm0iOgogICAgICAgICAgICByZXBseSA9IHJ1bl9waWNrX2NvbmZpcm0oX2NsaWVudCwgaGlzdG9yeSwgdlsic2VsZWN0ZWRfdXNlX2Nhc2UiXSwgdlsicm9sZSJdLCB2WyJsYW5ndWFnZSJdKQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlcGx5LCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0id2luX29wZW4iLCBuZWVkc191c2VyX2lucHV0PUZhbHNlKQoKICAgICAgICBpZiByZXEuc3RhZ2UgPT0gIndpbl9vcGVuIjoKICAgICAgICAgICAgcmVwbHkgPSBydW5fd2luX29wZW4oX2NsaWVudCwgaGlzdG9yeSwgdlsic2VsZWN0ZWRfdXNlX2Nhc2UiXSwgdlsicm9sZSJdLCB2WyJsYW5ndWFnZSJdKQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlcGx5LCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0id2luX2V4ZWN1dGUiLCBuZWVkc191c2VyX2lucHV0PVRydWUpCgogICAgICAgIGlmIHJlcS5zdGFnZSA9PSAid2luX2V4ZWN1dGUiOgogICAgICAgICAgICByZXN1bHQgPSBydW5fd2luX2V4ZWN1dGUoCiAgICAgICAgICAgICAgICBfY2xpZW50LCBoaXN0b3J5LCB2WyJzZWxlY3RlZF91c2VfY2FzZSJdLAogICAgICAgICAgICAgICAgdi5nZXQoInVzZXJfdGFza19kZXRhaWxzIiwgcmVxLnVzZXJfbWVzc2FnZSksCiAgICAgICAgICAgICAgICB2WyJyb2xlIl0sIHZbImxhbmd1YWdlIl0KICAgICAgICAgICAgKQogICAgICAgICAgICB2WyJ0YXNrX291dHB1dCJdID0gcmVzdWx0WyJvdXRwdXQiXQogICAgICAgICAgICB2WyJ0YXNrX291dHB1dF9zdW1tYXJ5Il0gPSByZXN1bHRbInN1bW1hcnkiXQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlc3VsdFsib3V0cHV0Il0sIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJ3aW5fY29uZmlybSIsIG5lZWRzX3VzZXJfaW5wdXQ9VHJ1ZSkKCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJ3aW5fY29uZmlybSI6CiAgICAgICAgICAgIHJlcGx5ID0gcnVuX3dpbl9jb25maXJtKF9jbGllbnQsIGhpc3RvcnksIHZbImxhbmd1YWdlIl0pCiAgICAgICAgICAgIHJldHVybiBDaGF0UmVzcG9uc2UocmVwbHk9cmVwbHksIHZhcmlhYmxlcz12LCBuZXh0X3N0YWdlPSJwaWxsIiwgbmVlZHNfdXNlcl9pbnB1dD1GYWxzZSkKCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJwaWxsIjoKICAgICAgICAgICAgcGlsbF9pZCA9IHNlbGVjdF9waWxsKHZbInNlbGVjdGVkX3VzZV9jYXNlIl0pCiAgICAgICAgICAgIHZbInBpbGxfaWQiXSA9IHBpbGxfaWQKICAgICAgICAgICAgcmVwbHkgPSBydW5fcGlsbCgKICAgICAgICAgICAgICAgIF9jbGllbnQsIGhpc3RvcnksIHBpbGxfaWQsIHZbInNlbGVjdGVkX3VzZV9jYXNlIl0sCiAgICAgICAgICAgICAgICB2WyJyb2xlIl0sIHZbImxhbmd1YWdlIl0sIHYuZ2V0KCJ0YXNrX291dHB1dF9zdW1tYXJ5IiwgIiIpCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmV0dXJuIENoYXRSZXNwb25zZShyZXBseT1yZXBseSwgdmFyaWFibGVzPXYsIG5leHRfc3RhZ2U9Im1hcCIsIG5lZWRzX3VzZXJfaW5wdXQ9VHJ1ZSkKCiAgICAgICAgaWYgcmVxLnN0YWdlID09ICJtYXAiOgogICAgICAgICAgICByZXBseSA9IHJ1bl9tYXAoX2NsaWVudCwgaGlzdG9yeSwgdlsicm9sZSJdLCB2WyJzZWxlY3RlZF91c2VfY2FzZSJdLCB2LmdldCgicGlsbF9pZCIsIDEpLCB2WyJsYW5ndWFnZSJdKQogICAgICAgICAgICByZXR1cm4gQ2hhdFJlc3BvbnNlKHJlcGx5PXJlcGx5LCB2YXJpYWJsZXM9diwgbmV4dF9zdGFnZT0iZG9uZSIsIG5lZWRzX3VzZXJfaW5wdXQ9RmFsc2UpCgogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHJhaXNlIEhUVFBFeGNlcHRpb24oc3RhdHVzX2NvZGU9NTAwLCBkZXRhaWw9c3RyKGUpKQo="
with open("server.py", "w", encoding="utf-8") as f:
    f.write(base64.b64decode(_srv_b64).decode("utf-8"))

print("server files written: chispa_core.py, server.py")


In [ ]:
# Cell 8: Start FastAPI server
import subprocess, time

server_process = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(2)
print("FastAPI server started on port 8000.")


In [ ]:
# Cell 9: ngrok tunnel
!pip install pyngrok -q
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient
import os

ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
ngrok.set_auth_token(ngrok_token)

public_url = ngrok.connect(8000)
os.environ["CHISPA_PUBLIC_URL"] = str(public_url)
print(f"Chispa is live at: {public_url}")


In [ ]:
# Cell 10: Inject ngrok URL into index.html
import os, base64

if not os.path.exists("index.html"):
    print("index.html not found — regenerating...")
    _b64 = "..."
    html_content = base64.b64decode(_b64).decode("utf-8")
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html_content)
    print("index.html created.")

with open("index.html", "r", encoding="utf-8") as f:
    html = f.read()

ngrok_url = os.environ.get("CHISPA_PUBLIC_URL", "")
if not ngrok_url:
    print("WARNING: CHISPA_PUBLIC_URL not set — URL injection skipped")
else:
    html = html.replace(
        "window.CHISPA_API_URL = null",
        f"window.CHISPA_API_URL = '{ngrok_url}/api/chat'"
    )
    with open("index.html", "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Injected API URL: {ngrok_url}/api/chat")
